In [ ]:
import pandas as pd
import numpy as np
import tkinter as tk
import matplotlib.pyplot as plt
import time

from pathlib import Path
from tkinter import filedialog
from scipy.stats import gaussian_kde
from sklearn.model_selection import KFold
from sklearn.neighbors import KernelDensity


In [ ]:
def load_dataframe_from_path(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

    ext = path.suffix.lower()

    if ext == ".csv":
        df = pd.read_csv(path, delimiter=",")
        
    elif ext in [".otus", ".txt", ".meta", ".taxonomy"]:
        df = pd.read_csv(path, sep="\t")
        
    else:
        raise ValueError(
            f"Formato no soportado: {ext}. "
            "Usa archivos .csv, .otus, .txt, .meta o .taxonomy"
        )

    return df


def load_multiple_dataframes():
    root = tk.Tk()
    root.withdraw()  
    root.attributes("-topmost", True)  

    file_paths = filedialog.askopenfilenames(
        title="Selecciona uno o varios datasets",
        filetypes=[
            ("Archivos soportados", "*.csv *.otus *.txt *.meta *.taxonomy"),
            ("CSV", "*.csv"),
            ("OTUS", "*.otus"),
            ("TXT", "*.txt"),
            ("META", "*.meta"),
            ("TAXONOMY", "*.taxonomy"),
            ("Todos los archivos", "*.*")
        ]
    )

    if not file_paths:
        print("No se seleccionó ningún archivo.")
        return {}

    dataframes = {}

    for file_path in file_paths:
        path = Path(file_path)
        try:
            df = load_dataframe_from_path(path)
            dataframes[path.stem] = df
            print(f"Cargado: {path.name} -> shape {df.shape}")
        except Exception as e:
            print(f"Error al cargar {path.name}: {e}")

    return dataframes

In [ ]:
dfs = load_multiple_dataframes()

if not dfs:
    print("No se cargó ningún archivo.")
else:
    for nombre, df in dfs.items():
        print(f"\nArchivo cargado: {nombre}")
        print(f"Shape del DataFrame: {df.shape}")
        print(f"Columnas: {df.columns.tolist()[:10]}{' ...' if len(df.columns) > 10 else ''}")

In [ ]:
def get_otu_positive_values(dfs, data_df_name="otu_data_converted"):
    if data_df_name not in dfs:
        raise KeyError(f"No existe '{data_df_name}' en dfs. Disponibles: {list(dfs.keys())}")

    df = dfs[data_df_name].copy()
    values = df.select_dtypes(include=[np.number]).to_numpy(dtype=float).ravel()
    values = values[np.isfinite(values)]
    positives = values[values > 0]

    if positives.size == 0:
        raise ValueError(f"'{data_df_name}' no contiene valores numericos positivos.")

    return positives, df.shape


def suggest_grid_size(n_values):
    if n_values <= 250_000:
        return 1000
    if n_values <= 1_000_000:
        return 1500
    return 2000


def make_log_grid(values, grid_size, bandwidth):
    positives = np.asarray(values, dtype=float).ravel()
    positives = positives[np.isfinite(positives)]
    positives = positives[positives > 0]

    if positives.size == 0:
        raise ValueError("Se requiere al menos un valor positivo.")
    if bandwidth <= 0:
        raise ValueError("El bandwidth debe ser positivo.")

    lower = max(float(np.min(positives)) * 1e-3, 1e-12)
    upper = float(np.max(positives) + 8.0 * bandwidth)
    return np.logspace(np.log10(lower), np.log10(upper), int(grid_size))


In [ ]:
def get_kernels():
    return ("gaussian", "epanechnikov", "tophat", "exponential", "linear", "cosine")


def get_color_map():
    return {
        "gaussian": "#4c78a8",
        "epanechnikov": "#e45756",
        "tophat": "#f58518",
        "exponential": "#54a24b",
        "linear": "#b279a2",
        "cosine": "#9d755d",
    }


def positive_mass(pdf, x_grid):
    return float(np.trapezoid(pdf, x_grid))


def normalize_conditional(pdf, x_grid):
    mass = positive_mass(pdf, x_grid)

    if not np.isfinite(mass) or mass <= 0:
        raise ValueError("La densidad KDE tiene masa no positiva o no finita.")

    return pdf / mass, mass


def mode_kde(pdf, x_grid):
    return float(x_grid[int(np.argmax(pdf))])


def evaluate_kde_gaussian_fast(data, x_grid, bandwidth):
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth, breadth_first=False)
    kde.fit(np.asarray(data, dtype=float).reshape(-1, 1))
    return np.exp(kde.score_samples(np.asarray(x_grid, dtype=float).reshape(-1, 1)))


def sorted_prefix(data):
    sorted_data = np.sort(np.asarray(data, dtype=float))
    prefix_1 = np.concatenate(([0.0], np.cumsum(sorted_data)))
    prefix_2 = np.concatenate(([0.0], np.cumsum(sorted_data * sorted_data)))
    return sorted_data, prefix_1, prefix_2


def evaluate_finite_support_fast(data, x_grid, bandwidth, kernel):
    sorted_data, prefix_1, prefix_2 = sorted_prefix(data)
    n = float(sorted_data.size)
    x = np.asarray(x_grid, dtype=float)
    left = np.searchsorted(sorted_data, x - bandwidth, side="left")
    right = np.searchsorted(sorted_data, x + bandwidth, side="right")
    count = (right - left).astype(float)

    if kernel == "tophat":
        return 0.5 * count / (n * bandwidth)

    if kernel == "epanechnikov":
        sum_1 = prefix_1[right] - prefix_1[left]
        sum_2 = prefix_2[right] - prefix_2[left]
        sq_dist = count * x * x - 2.0 * x * sum_1 + sum_2
        return 0.75 * (count - sq_dist / (bandwidth * bandwidth)) / (n * bandwidth)

    if kernel == "linear":
        mid = np.searchsorted(sorted_data, x, side="right")
        mid = np.minimum(np.maximum(mid, left), right)
        left_count = (mid - left).astype(float)
        right_count = (right - mid).astype(float)
        left_sum = prefix_1[mid] - prefix_1[left]
        right_sum = prefix_1[right] - prefix_1[mid]
        abs_dist = x * left_count - left_sum + right_sum - x * right_count
        return (count - abs_dist / bandwidth) / (n * bandwidth)

    if kernel == "cosine":
        c = np.pi / (2.0 * bandwidth)
        prefix_cos = np.concatenate(([0.0], np.cumsum(np.cos(c * sorted_data))))
        prefix_sin = np.concatenate(([0.0], np.cumsum(np.sin(c * sorted_data))))
        sum_cos = prefix_cos[right] - prefix_cos[left]
        sum_sin = prefix_sin[right] - prefix_sin[left]
        values = np.cos(c * x) * sum_cos + np.sin(c * x) * sum_sin
        return (np.pi / 4.0) * values / (n * bandwidth)

    raise ValueError(f"Kernel no soportado por el metodo rapido: {kernel}")


def evaluate_exponential_fast(data, x_grid, bandwidth):
    sorted_data = np.sort(np.asarray(data, dtype=float))
    n = float(sorted_data.size)
    x = np.asarray(x_grid, dtype=float)
    order = np.argsort(x)
    xs = x[order]

    left_sum = np.zeros_like(xs)
    acc = 0.0
    data_idx = 0
    previous_x = xs[0]

    for i, current_x in enumerate(xs):
        if i > 0:
            acc *= np.exp(-(current_x - previous_x) / bandwidth)
        while data_idx < sorted_data.size and sorted_data[data_idx] <= current_x:
            acc += np.exp((sorted_data[data_idx] - current_x) / bandwidth)
            data_idx += 1
        left_sum[i] = acc
        previous_x = current_x

    right_sum = np.zeros_like(xs)
    acc = 0.0
    data_idx = sorted_data.size - 1
    previous_x = xs[-1]

    for i in range(xs.size - 1, -1, -1):
        current_x = xs[i]
        if i < xs.size - 1:
            acc *= np.exp(-(previous_x - current_x) / bandwidth)
        while data_idx >= 0 and sorted_data[data_idx] > current_x:
            acc += np.exp((current_x - sorted_data[data_idx]) / bandwidth)
            data_idx -= 1
        right_sum[i] = acc
        previous_x = current_x

    out_sorted = 0.5 * (left_sum + right_sum) / (n * bandwidth)
    out = np.empty_like(out_sorted)
    out[order] = out_sorted
    return out


def evaluate_kde_fast(data, x_grid, bandwidth, kernel):
    finite_fast_kernels = {"epanechnikov", "tophat", "linear", "cosine"}
    data = np.asarray(data, dtype=float).ravel()
    data = data[np.isfinite(data)]

    if bandwidth <= 0:
        raise ValueError("El bandwidth debe ser positivo.")
    if kernel not in get_kernels():
        raise ValueError(f"Kernel desconocido: {kernel}")
    if kernel in finite_fast_kernels:
        return evaluate_finite_support_fast(data, x_grid, bandwidth, kernel)
    if kernel == "exponential":
        return evaluate_exponential_fast(data, x_grid, bandwidth)

    return evaluate_kde_gaussian_fast(data, x_grid, bandwidth)


In [ ]:
def evaluate_densities(values, grid_size, bandwidths):
    densities = {}
    rows = []

    for kernel in get_kernels():
        bandwidth = bandwidths[kernel]
        x_grid = make_log_grid(values, grid_size, bandwidth)
        pdf = evaluate_kde_fast(values, x_grid, bandwidth, kernel)
        density, mass = normalize_conditional(pdf, x_grid)
        densities[kernel] = (x_grid, density)
        rows.append({
            "kernel": kernel,
            "bandwidth": bandwidth,
            "masa": mass,
            "moda_kde": mode_kde(density, x_grid),
        })

    return densities, pd.DataFrame(rows)


def plot_all_kernels(densities, bandwidths, title):
    color_map = get_color_map()
    fig, ax = plt.subplots(figsize=(10, 5))

    for kernel in get_kernels():
        x_grid, density = densities[kernel]
        ax.plot(
            x_grid,
            density,
            color=color_map[kernel],
            linewidth=1.8,
            label=f"{kernel} (h={bandwidths[kernel]:.3g})",
        )

    ax.set_xscale("log")
    ax.set_xlabel("Valor OTU positivo")
    ax.set_ylabel("Densidad condicional")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend(ncol=2, fontsize=9)
    fig.tight_layout()
    plt.show()


def plot_kernel_grid(densities, bandwidths, title):
    color_map = get_color_map()
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)

    for ax, kernel in zip(axes.ravel(), get_kernels()):
        x_grid, density = densities[kernel]
        ax.plot(x_grid, density, color=color_map[kernel], linewidth=1.8)
        ax.set_xscale("log")
        ax.set_title(f"{kernel} | h={bandwidths[kernel]:.3g}")
        ax.grid(True, alpha=0.25)

    for ax in axes[-1, :]:
        ax.set_xlabel("Valor OTU positivo")
    for ax in axes[:, 0]:
        ax.set_ylabel("Densidad condicional")

    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.show()


def kernel_calculation_from_loaded(
    dfs,
    data_df_name="otu_data_converted",
    grid_size=1000,
    common_bandwidth=129.874938044,
    kernel_bandwidths=None,
    test_kernel_bandwidths=None,
    verbose=True
):
    if kernel_bandwidths is None:
        kernel_bandwidths = {
            "gaussian": 129.874938044,
            "epanechnikov": 1.77134956169,
            "tophat": 1.77134956169,
            "exponential": 15.167528295,
            "linear": 1.77134956169,
            "cosine": 1.77134956169,
        }

    if test_kernel_bandwidths is None:
        test_kernel_bandwidths = kernel_bandwidths.copy()

    values, data_shape = get_otu_positive_values(dfs, data_df_name=data_df_name)

    data_summary = pd.DataFrame([{
        "filas": data_shape[0],
        "columnas": data_shape[1],
        "valores_positivos": len(values),
        "grid_size": grid_size,
        "bandwidth_comun": common_bandwidth,
    }])

    if verbose:
        print(f"DataFrame analizado: {data_df_name}")
        print(f"Valores positivos: {len(values)}")
        print(f"Grid usado: {grid_size}")
        print(f"Bandwidth comun: {common_bandwidth}")

    display(data_summary)

    common_bandwidths = {kernel: common_bandwidth for kernel in get_kernels()}
    common_densities, common_summary = evaluate_densities(values, grid_size, common_bandwidths)
    display(common_summary)

    plot_all_kernels(common_densities, common_bandwidths, f"KDE de los seis kernels (h comun={common_bandwidth:.3g})")
    plot_kernel_grid(common_densities, common_bandwidths, "KDE por kernel")

    best_densities, best_summary = evaluate_densities(values, grid_size, kernel_bandwidths)
    display(best_summary)
    plot_all_kernels(best_densities, kernel_bandwidths, "KDE de los seis kernels con h particular")

    test_densities, test_summary = evaluate_densities(values, grid_size, test_kernel_bandwidths)
    display(test_summary)
    plot_kernel_grid(test_densities, test_kernel_bandwidths, "Pruebas particulares de bandwidth por kernel")

    result = {
        "values": values,
        "common_densities": common_densities,
        "best_densities": best_densities,
        "test_densities": test_densities,
        "common_summary": common_summary,
        "best_summary": best_summary,
        "test_summary": test_summary,
    }

    return result


In [ ]:
kernel_outputs = kernel_calculation_from_loaded(
    dfs=dfs,
    data_df_name="otu_data_converted",
    grid_size=1000,
    common_bandwidth=129.874938044,
    kernel_bandwidths={
        "gaussian": 129.874938044,
        "epanechnikov": 1.77134956169,
        "tophat": 1.77134956169,
        "exponential": 15.167528295,
        "linear": 1.77134956169,
        "cosine": 1.77134956169,
    },
    test_kernel_bandwidths={
        "gaussian": 129.874938044,
        "epanechnikov": 1.77134956169,
        "tophat": 1.77134956169,
        "exponential": 15.167528295,
        "linear": 1.77134956169,
        "cosine": 1.77134956169,
    },
    verbose=True,
)
